# 第14章 矩阵运算与特征分解——协方差矩阵的密码

> **动机先行**: 第13章教会了我们用向量"表示"多资产世界——一个权重向量 $\mathbf{w}$、一个收益率向量 $\mathbf{r}$、一个收益率矩阵 $\mathbf{R}$ 的"列视角"。但真正的量化工作远不止"表示"，你需要**操作**这些数学对象：同时回测5个策略的组合净值、从协方差矩阵中提取"风险的主方向"、计算最小方差组合的精确权重。这些操作的数学基础，正是本章要讲的**矩阵乘法、逆矩阵、特征值分解**——它们构成了量化金融计算栈的"汇编语言"层。
>
> **量化实战定位**: 本章的每一个数学概念都直接对应一个量化工作流——矩阵乘法对应批量回测、逆矩阵对应组合优化、特征分解对应PCA风险模型。读完本章后，你将能看懂专业量化系统的核心计算逻辑，并能用NumPy独立实现这些操作。

---

## 14.1 动机: 从"一个组合"到"一批组合"——矩阵乘法的量化驱动

第13章的高潮是矩阵-向量乘法 $\mathbf{R}\mathbf{w}$: 用一组权重 $\mathbf{w}$ 去"加权"收益率矩阵 $\mathbf{R}$，一次性算出组合的全部历史收益率。

但量化实践中，你很少只做**一个**组合。典型场景是:

- **策略比较**: 同时跑等权、市值加权、最小方差、风险平价、动量多空5个策略，看谁的夏普比率最高
- **参数扫描**: 对同一个因子策略，测试10组不同的调仓频率(5天、10天、20天、...)
- **情景分析**: 对同一组合计算在3种不同协方差矩阵估计(样本、收缩、因子模型)下的风险

如果每组权重都写一个 $\mathbf{R}\mathbf{w}$，你的代码会变成:

In [ ]:
rp_eq  = R @ w_eq      # 等权
rp_mv  = R @ w_mv      # 最小方差
rp_rp  = R @ w_rp      # 风险平价
rp_mom = R @ w_mom     # 动量多空

这没问题——但矩阵乘法可以做得更优雅。把5组权重向量**并排**成一个 $N \times 5$ 的矩阵 $\mathbf{W}$:

$$\mathbf{W} = \begin{bmatrix}
| & | & | & | & | \\
\mathbf{w}_{\text{eq}} & \mathbf{w}_{\text{mv}} & \mathbf{w}_{\text{rp}} & \mathbf{w}_{\text{mom}} & \mathbf{w}_{\text{opt}} \\
| & | & | & | & |
\end{bmatrix}$$

然后**一次矩阵乘法** $\mathbf{R}\mathbf{W}$ 就得到了全部5条组合收益率序列。$\mathbf{R}$ 是 $T \times N$，$\mathbf{W}$ 是 $N \times 5$，结果是 $T \times 5$——每一列是一种策略的组合收益率时间序列。

这就是本章的第一个主题——**矩阵乘法**: 不是抽象的"行列点积规则"，而是"批量线性组合的计算引擎"。

第二个主题则更进一步。如果说矩阵乘法回答了"怎么算"，那么**特征值分解**回答了"长什么样"。协方差矩阵 $\mathbf{\Sigma}$ 是一个 $N \times N$ 的对称矩阵，里面装了 $N(N+1)/2$ 个协方差数字。但其中"真正重要的信息"远少于这个数字——特征值分解就是在做这种"信息压缩": 用前 $K$ 个特征向量和特征值，逼近整个协方差矩阵的结构。这是 PCA (主成分分析) 风险模型的数学基础。

> **量化人的视角**: 矩阵乘法 = 策略回测的循环展开; 特征分解 = 风险结构的X光透视。

---

## 14.2 矩阵乘法: 线性变换的计算语言

### 14.2.1 矩阵乘法的定义

设 $\mathbf{A}$ 是 $m \times n$ 矩阵，$\mathbf{B}$ 是 $n \times p$ 矩阵，它们的乘积 $\mathbf{C} = \mathbf{A}\mathbf{B}$ 是 $m \times p$ 矩阵，其中:

$$\boxed{C_{ij} = \sum_{k=1}^{n} A_{ik} B_{kj}}$$

简单说: $\mathbf{C}$ 的 $(i, j)$ 元素 = $\mathbf{A}$ 的第 $i$ 行 与 $\mathbf{B}$ 的第 $j$ 列 的**点积**。

**维度约束**: $\mathbf{A}$ 的列数必须等于 $\mathbf{B}$ 的行数 ($n$)。结果矩阵的维度是"$\mathbf{A}$ 的行数 $\times$ $\mathbf{B}$ 的列数"。

一个具体例子:

$$\mathbf{A} = \begin{pmatrix} 1 & 2 \\ 3 & 4 \end{pmatrix}, \quad \mathbf{B} = \begin{pmatrix} 5 & 6 \\ 7 & 8 \end{pmatrix}$$

$$\mathbf{A}\mathbf{B} = \begin{pmatrix}
1 \times 5 + 2 \times 7 & 1 \times 6 + 2 \times 8 \\
3 \times 5 + 4 \times 7 & 3 \times 6 + 4 \times 8
\end{pmatrix} = \begin{pmatrix} 19 & 22 \\ 43 & 50 \end{pmatrix}$$

### 14.2.2 矩阵乘法的"列视角": 最重要的量化直觉

第13章教过: 矩阵-向量乘法 $\mathbf{R}\mathbf{w}$ = 以 $\mathbf{w}$ 的分量为权重，对 $\mathbf{R}$ 的各列做线性组合。

矩阵-矩阵乘法 $\mathbf{R}\mathbf{W}$ 是这个直觉的直接推广:

$$\boxed{\mathbf{R}\mathbf{W} = \begin{bmatrix}
\mathbf{R}\mathbf{w}_1 & \mathbf{R}\mathbf{w}_2 & \cdots & \mathbf{R}\mathbf{w}_p
\end{bmatrix}}$$

其中 $\mathbf{w}_j$ 是 $\mathbf{W}$ 的第 $j$ 列。**$\mathbf{R}\mathbf{W}$ 的第 $j$ 列，等于 $\mathbf{R}$ 乘以 $\mathbf{W}$ 的第 $j$ 列的矩阵-向量积。**

换句话说: $\mathbf{R}\mathbf{W}$ 就是把矩阵-向量乘法 $\mathbf{R}\mathbf{w}$ 对 $\mathbf{W}$ 的每一列**独立执行**，然后将结果并排在一起。

**量化含义——批量回测的核心**:

$\mathbf{R}$ 是 $T \times N$ 的收益率矩阵，$\mathbf{W}$ 是 $N \times K$ 的权重矩阵(每列一种策略)，则:

$$\mathbf{R}_{p} = \mathbf{R}\mathbf{W}$$

$\mathbf{R}_p$ 是 $T \times K$ 的组合收益率矩阵——第 $j$ 列是第 $j$ 种策略的组合收益率序列。**一次矩阵乘法，完成了 $K$ 个策略的全时期回测。**

> **计算效率注记**: 在实践中，NumPy 的 `R @ W` 会调用高度优化的 BLAS (Basic Linear Algebra Subprograms) 库。对于 $T=1210, N=50, K=5$ 这样的规模，矩阵乘法比 Python 循环快几十到几百倍。当 $N$ 扩展到 500 或 5000 时，差距更是指数级的——这也是为什么量化系统重度依赖线性代数库。

### 14.2.3 矩阵乘法的运算性质

矩阵乘法**不满足交换律**，但满足以下关键性质:

| 性质 | 公式 | 量化含义 |
|------|------|---------|
| **结合律** | $(\mathbf{A}\mathbf{B})\mathbf{C} = \mathbf{A}(\mathbf{B}\mathbf{C})$ | 先算因子收益还是先算组合权重，最终结果一致 |
| **分配律** | $\mathbf{A}(\mathbf{B} + \mathbf{C}) = \mathbf{A}\mathbf{B} + \mathbf{A}\mathbf{C}$ | 基准+超额收益可以分开归因再相加 |
| **与转置的关系** | $(\mathbf{A}\mathbf{B})^T = \mathbf{B}^T \mathbf{A}^T$ | 见14.3节 |
| **非交换性** | $\mathbf{A}\mathbf{B} \neq \mathbf{B}\mathbf{A}$ (一般情况) | 权重矩阵 × 收益率矩阵 ≠ 收益率矩阵 × 权重矩阵 |

> **注意**: 在一般矩阵乘法中，$\mathbf{A}\mathbf{B}$ 和 $\mathbf{B}\mathbf{A}$ 甚至在维度上都不一定兼容。即使两者都是方阵，交换律通常也不成立。这个"不可交换性"在金融中有深层含义: 先选股再调仓 ≠ 先调仓再选股。

### 14.2.4 因子收益归因: 矩阵乘法的经典应用

Fama-French 三因子模型是矩阵乘法最直接的金融应用。设 $\mathbf{B}$ 是 $N \times 3$ 的因子暴露矩阵 (每行一只股票，3列对应市场 MKT、市值 SMB、价值 HML 的暴露系数)，$\mathbf{f}_t$ 是第 $t$ 期的 $3 \times 1$ 因子收益率向量，则第 $t$ 期 $N$ 只股票中能被因子解释的收益部分为:

$$\mathbf{r}_t^{\text{factor}} = \mathbf{B} \mathbf{f}_t \quad (\text{矩阵-向量乘法, } N \times 3 \cdot 3 \times 1 = N \times 1)$$

把 $T$ 期的因子收益率纵向堆叠成 $T \times 3$ 矩阵 $\mathbf{F}$ (每行一期)，则全时期 $T \times N$ 的**因子解释收益矩阵**一次矩阵乘法即可得到:

$$\boxed{\mathbf{R}_{\text{factor}} = \mathbf{F} \mathbf{B}^T} \quad (\text{矩阵-矩阵乘法, } T \times 3 \cdot 3 \times N = T \times N)$$

注意 $\mathbf{B}^T$ 的转置: $\mathbf{B}$ 是 $N \times 3$，$\mathbf{B}^T$ 是 $3 \times N$，这样才能与 $T \times 3$ 的 $\mathbf{F}$ 相容相乘。

**特异收益 (Idiosyncratic Return)**——因子无法解释的部分——由实际收益减去因子收益得到:

$$\mathbf{E} = \mathbf{R} - \mathbf{F} \mathbf{B}^T$$

其中 $\mathbf{R}$ 是 $T \times N$ 的实际收益率矩阵。$\mathbf{E}$ 的每一列是该股票的**特异收益时间序列**——这是统计套利和"市场中性"策略的 Alpha 来源: 做多正特异收益、做空负特异收益，同时对冲掉因子暴露。

**一个微型计算示例**。取3只股票、2个因子(市场+规模)、最近3个交易日，用矩阵乘法验证整个计算流程:

In [ ]:
import numpy as np

# 因子暴露矩阵 B (3只股票 × 2个因子: MKT, SMB)
B = np.array([[1.2, -0.3],   # 股票1: beta_MKT=1.2, 大盘股(beta_SMB为负)
              [0.9,  0.5],   # 股票2: beta_MKT=0.9, 小盘股
              [1.0,  0.1]])  # 股票3: beta_MKT=1.0, 轻微小盘

# 因子收益率矩阵 F (3个交易日 × 2个因子)
F = np.array([[ 0.008,  0.003],   # 第1天: 市场涨0.8%, 小盘优于大盘0.3%
              [-0.005, -0.002],   # 第2天: 市场跌0.5%, 大盘优于小盘0.2%
              [ 0.012,  0.004]])  # 第3天: 市场涨1.2%, 小盘优于大盘0.4%

# 因子解释收益 R_factor = F @ B^T  (3×2 · 2×3 = 3×3)
R_factor = F @ B.T
print("因子解释收益矩阵 (每行=一天, 每列=一只股票):")
print(R_factor.round(5))
# 例: 第1天股票1的因子收益 = 1.2*0.008 + (-0.3)*0.003 = 0.0087

# 假设实际收益 (做一点扰动模拟特异收益)
R_actual = R_factor + np.random.normal(0, 0.003, size=R_factor.shape)
E = R_actual - R_factor  # 特异收益矩阵 (3×3)
print(f"\n特异收益矩阵 E = R - F@B^T (形状: {E.shape}):")
print(E.round(5))

# 验证维度: F(T×K), B^T(K×N) → R_factor(T×N)
print(f"\nF 形状: {F.shape}, B^T 形状: {B.T.shape}, R_factor 形状: {R_factor.shape}")

**运行结果**:
```
因子解释收益矩阵 (每行=一天, 每列=一只股票):
[[ 0.0087  0.0087  0.0083]
 [-0.0054 -0.0055 -0.0052]
 [ 0.0132  0.0128  0.0124]]

特异收益矩阵 E = R - F@B^T (形状: (3, 3)):
[[ 0.0012 -0.0008  0.0003]
 [-0.0015  0.0021 -0.0011]
 [ 0.0007 -0.0017  0.0025]]

F 形状: (3, 2), B^T 形状: (2, 3), R_factor 形状: (3, 3)
```

**解读**: 第一天的因子收益矩阵显示，所有三只股票因子部分都为正(因为当天市场涨了0.8%)，但股票3略低(因为它的市场暴露=1.0, 低于股票1的1.2)。实际收益与因子收益的差(特异收益矩阵 $\mathbf{E}$)就是量化策略尝试预测和交易的部分。

> **量化实践——Barra 模型的矩阵视角**: 在 Barra 等商业风险模型中，因子暴露矩阵 $\mathbf{B}$ 可能有 50-70 个因子(行业哑变量 + 风格因子)。$\mathbf{F} \mathbf{B}^T$ 算出的 $\mathbf{R}_{\text{factor}}$ 是"能被共同因子解释的收益部分"，$\mathbf{E} = \mathbf{R} - \mathbf{F} \mathbf{B}^T$ 是特异收益。风险模型的核心假设是: **特异收益在不同股票之间不相关**($Cov(\varepsilon_i, \varepsilon_j)=0, i \neq j$)，这使得总协方差矩阵可以分解为"低秩因子部分 + 对角特异部分"——这正是14.3.3节的结构。

### 14.2.5 量化实战: 批量回测——一个矩阵乘法替代K个循环

理论讲完了，上真实数据。从 `stock_data_50` 中选10只不同行业的股票，构建5种策略（等权、主动超配、波动率倒数、价格加权、多空），一次 `R @ W` 算出全部净值曲线:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False

# 加载数据, 选取10只不同行业的股票
selected = ['000002.SZ', '600519.SH', '300750.SZ', '000858.SZ',
            '601398.SH', '002415.SZ', '000725.SZ', '300059.SZ',
            '002230.SZ', '600030.SH']
csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

df_recent = df[df['time'] >= '2024-06-01'].copy()
pivot_close = df_recent.pivot(index='time', columns='thscode', values='close')[selected]
log_rets = np.log(pivot_close / pivot_close.shift(1)).dropna()

R = log_rets.values  # (T, 10)
T, N = R.shape
dates = log_rets.index

print(f"收益率矩阵 R: {T} 天 x {N} 只股票")

# 构建5组策略的权重矩阵 W (N x 5)
w_eq = np.ones(N) / N                                      # 策略1: 等权
w_active = np.array([0.15,0.20,0.20,0.15,0.05,0.05,0.05,0.05,0.05,0.05])  # 策略2: 超配
vols = np.std(R, axis=0)                                   # 策略3: 波动率倒数
w_invvol = (1.0 / vols) / np.sum(1.0 / vols)
avg_prices = pivot_close.mean().values                     # 策略4: 价格加权(市值代理)
w_cap = avg_prices / np.sum(avg_prices)
w_ls = np.zeros(N); vol_rank = np.argsort(vols)            # 策略5: 多空
w_ls[vol_rank[:3]] = 0.25; w_ls[vol_rank[-3:]] = -0.25

W = np.column_stack([w_eq, w_active, w_invvol, w_cap, w_ls])
strategy_names = ['等权', '主动超配', '波动率倒数', '价格加权', '多空(低-高波动)']

# 核心: 一次矩阵乘法, 批量回测全部策略!
Rp = R @ W  # (T, 5)

# 绩效指标
ann_vol = np.std(Rp, axis=0) * np.sqrt(252)
ann_ret = np.mean(Rp, axis=0) * 252
sharpe = ann_ret / ann_vol
cum_ret = np.exp(np.cumsum(Rp, axis=0))
max_dd = np.array([np.min(cum_ret[:,j] / np.maximum.accumulate(cum_ret[:,j]) - 1)
                   for j in range(5)])

print(f"\n{'策略':<16} {'年化收益':>8} {'年化波动':>8} {'夏普比率':>8} {'最大回撤':>8}")
print("-" * 52)
for j, name in enumerate(strategy_names):
    print(f"{name:<16} {ann_ret[j]:>8.2%} {ann_vol[j]:>8.2%} {sharpe[j]:>8.2f} {max_dd[j]:>8.2%}")

# 画净值曲线
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors = ['#2980B9', '#E74C3C', '#27AE60', '#8E44AD', '#E67E22']
for j in range(5):
    axes[0].plot(dates, cum_ret[:, j], label=strategy_names[j],
                 color=colors[j], linewidth=1.3, alpha=0.85)
axes[0].set_title('五种策略净值曲线 (一次矩阵乘法 R@W 完成)', fontsize=13)
axes[0].set_xlabel('日期'); axes[0].set_ylabel('累计净值')
axes[0].legend(fontsize=8, loc='best'); axes[0].grid(True, alpha=0.3)

bars = axes[1].bar(strategy_names, sharpe, color=colors, alpha=0.8, edgecolor='black')
axes[1].axhline(y=0, color='gray', linewidth=0.8)
axes[1].set_title('各策略夏普比率对比', fontsize=13)
axes[1].set_ylabel('夏普比率')
axes[1].set_xticklabels(strategy_names, rotation=30, ha='right', fontsize=9)
for bar, s in zip(bars, sharpe):
    axes[1].text(bar.get_x()+bar.get_width()/2., bar.get_height(),
                 f'{s:.2f}', ha='center', va='bottom', fontsize=10)
axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/ch14_fig2_batch_backtest.png', dpi=150, bbox_inches='tight')
plt.show()

**运行结果**:
```
收益率矩阵 R: 481 天 x 10 只股票

策略                年化收益     年化波动     夏普比率     最大回撤
----------------------------------------------------
等权                  5.31%    21.98%     0.24   -18.47%
主动超配                1.10%    23.07%     0.05   -21.34%
波动率倒数               5.01%    19.69%     0.25   -15.76%
价格加权               -0.11%    23.19%    -0.00   -19.78%
多空(低-高波动)          -1.84%    20.82%    -0.09   -33.18%
```

> **关键收获**: `Rp = R @ W` 这一行代码，背后完成的是 $T \times N$ 乘 $N \times K$ 的矩阵乘法——$O(T \cdot N \cdot K)$ 次标量运算。如果在 Python 中写嵌套循环，对于 $T=481, N=10, K=5$ 需要三重循环24050次浮点乘加; 而 NumPy 调用 BLAS 在C/Fortran层完成，耗时几乎可以忽略。**当你管理的资产从10只扩展到500只时，这个差距就是"等5分钟"和"等0.01秒"的区别。**

---

## 14.3 转置与对称矩阵: 协方差矩阵的结构保证

### 14.3.1 转置的基本性质

第13章介绍了转置的基本定义 $( \mathbf{A}^T )_{ij} = A_{ji}$。这里补充两个与矩阵乘法相关的重要性质:

$$\boxed{(\mathbf{A}\mathbf{B})^T = \mathbf{B}^T \mathbf{A}^T}$$

$$\boxed{(\mathbf{A}^T)^T = \mathbf{A}}$$

第一个性质——"乘积的转置 = 转置的逆序乘积"——在量化推导中极其常用。例如，协方差矩阵的推导:

$$\mathbf{\Sigma} = \frac{1}{T-1} \tilde{\mathbf{R}}^T \tilde{\mathbf{R}}$$

其中 $\tilde{\mathbf{R}}$ 是去均值的收益率矩阵 ($T \times N$)。$\tilde{\mathbf{R}}^T \tilde{\mathbf{R}}$ 是一个 $N \times N$ 矩阵——这正是每对资产之间的协方差。

### 14.3.2 对称矩阵: 协方差矩阵的天然属性

如果 $\mathbf{A}^T = \mathbf{A}$，则 $\mathbf{A}$ 是**对称矩阵 (Symmetric Matrix)**。

协方差矩阵 $\mathbf{\Sigma}$ 必然是对称的——因为 $Cov(r_i, r_j) = Cov(r_j, r_i)$。从矩阵运算的角度:

$$\mathbf{\Sigma}^T = \left(\frac{1}{T-1} \tilde{\mathbf{R}}^T \tilde{\mathbf{R}}\right)^T = \frac{1}{T-1} \tilde{\mathbf{R}}^T (\tilde{\mathbf{R}}^T)^T = \frac{1}{T-1} \tilde{\mathbf{R}}^T \tilde{\mathbf{R}} = \mathbf{\Sigma}$$

> **量化实践——数值对称化**: 由于浮点运算的精度限制，你从 `numpy.cov()` 算出的矩阵可能不是精确对称的。在组合优化前，量化系统通常会强制执行 `Σ = (Σ + Σ.T) / 2` 以保证严格对称。这在调用 `scipy.optimize.minimize` 之类对对称性敏感的求解器时尤其重要。

### 14.3.3 协方差矩阵的"低秩+对角"结构

量化实践中，协方差矩阵通常被建模为:

$$\mathbf{\Sigma} = \mathbf{B} \mathbf{\Sigma}_f \mathbf{B}^T + \mathbf{D}$$

其中:
- $\mathbf{B}$: $N \times K$ 因子暴露矩阵
- $\mathbf{\Sigma}_f$: $K \times K$ 因子协方差矩阵 (通常远小于 $N$)
- $\mathbf{D}$: $N \times N$ 对角矩阵，表示特异方差

这个结构的计算优势在于: 求 $\mathbf{\Sigma}$ 的逆时，可以用 Woodbury 公式将 $O(N^3)$ 降为 $O(K^3)$——当 $N=5000, K=50$ 时，这是"能不能算"的区别。

---

## 14.4 逆矩阵: 组合优化与OLS的共同数学核心

### 14.4.1 逆矩阵的定义

对于 $n \times n$ 的方阵 $\mathbf{A}$，如果存在矩阵 $\mathbf{A}^{-1}$ 使得:

$$\boxed{\mathbf{A}\mathbf{A}^{-1} = \mathbf{A}^{-1}\mathbf{A} = \mathbf{I}}$$

则称 $\mathbf{A}^{-1}$ 为 $\mathbf{A}$ 的**逆矩阵 (Inverse Matrix)**。其中 $\mathbf{I}$ 是**单位矩阵 (Identity Matrix)**——对角线上全为1、其余为0的方阵，满足 $\mathbf{I}\mathbf{A} = \mathbf{A}\mathbf{I} = \mathbf{A}$。

并非所有矩阵都可逆。$\mathbf{A}$ 可逆的充要条件是: $\mathbf{A}$ 的各列线性无关 (即不存在一组不全为零的系数使列的线性组合为零向量)。

> **量化直觉**: 如果 $\mathbf{\Sigma}$ 的某两列高度相关 ($\rho \approx 0.99$)，则 $\mathbf{\Sigma}$ 接近"不可逆"——在数值计算中表现为条件数($\kappa$)极大。这就是第12章讲的多重共线性的矩阵版本: 协方差矩阵的条件数越大，组合优化的权重对输入参数的微小变化越敏感。

### 14.4.2 逆矩阵的金融应用

**应用1——最小方差组合**: Markowitz 均值-方差框架中，给定协方差矩阵 $\mathbf{\Sigma}$，在 $\mathbf{w}^T\mathbf{1}=1$ 约束下的最小方差组合权重为:

$$\boxed{\mathbf{w}_{\min} = \frac{\mathbf{\Sigma}^{-1}\mathbf{1}}{\mathbf{1}^T \mathbf{\Sigma}^{-1} \mathbf{1}}}$$

这个公式的推导(见练习题)需要用到拉格朗日乘子法，但它的结构是清晰的: $\mathbf{\Sigma}^{-1}\mathbf{1}$ 是"对全1向量做协方差逆变换"，分母则是归一化因子。

**应用2——OLS估计量**: 在第12章中，多元OLS的系数估计量为:

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

这里的 $(\mathbf{X}^T\mathbf{X})^{-1}$ 就是 $\mathbf{X}^T\mathbf{X}$ 这个 $k \times k$ 矩阵的逆。如果解释变量之间存在完全共线性，$\mathbf{X}^T\mathbf{X}$ 不可逆，OLS 无唯一解。

**应用3——精度矩阵与偏相关**: 协方差矩阵的逆 $\mathbf{\Sigma}^{-1}$ 称为**精度矩阵 (Precision Matrix)**。它的 $(i, j)$ 元素经过归一化后等于资产 $i$ 和 $j$ 的**偏相关系数**——即在控制了所有其他资产的影响后，两者之间的"净"相关性。这在图模型 (Graphical Lasso) 选股中有重要应用。

### 14.4.3 逆矩阵的运算性质

$$\boxed{(\mathbf{A}\mathbf{B})^{-1} = \mathbf{B}^{-1}\mathbf{A}^{-1}}$$

$$\boxed{(\mathbf{A}^{-1})^T = (\mathbf{A}^T)^{-1}}$$

注意逆序: 乘积的逆 = 逆的逆序乘积——与转置的性质类似。

> **数值实践**: 在量化代码中，你几乎**从不**显式计算 $\mathbf{\Sigma}^{-1}$。标准做法是用 `np.linalg.solve(Σ, 1)` 求解线性方程组 $\mathbf{\Sigma}\mathbf{w} = \mathbf{1}$。`solve` 在内部可能使用了 Cholesky 分解等更稳定的算法，精度远高于先求逆再乘向量。

### 14.4.4 量化实战: 最小方差组合——从公式到代码

公式 $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1} / (\mathbf{1}^T \mathbf{\Sigma}^{-1} \mathbf{1})$ 看起来很抽象，用代码实现只需三行:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

# 训练集: 用一年的数据估计协方差矩阵
df_train = df[(df['time'] >= '2024-06-01') & (df['time'] < '2025-06-01')].copy()
pivot_train = df_train.pivot(index='time', columns='thscode', values='close')
log_rets_train = np.log(pivot_train / pivot_train.shift(1)).dropna()
R_train = log_rets_train.values
T_train, N = R_train.shape

R_demean = R_train - R_train.mean(axis=0)
Sigma = (R_demean.T @ R_demean) / (T_train - 1)
Sigma_reg = Sigma + np.eye(N) * 1e-8               # 微弱正则化, 保证数值稳定

# 核心三行: 最小方差组合
ones = np.ones(N)
w_unnorm = np.linalg.solve(Sigma_reg, ones)         # 解 Σw = 1, 而非求 Σ^{-1}
w_minvar = w_unnorm / np.sum(w_unnorm)              # 归一化使权重和为1
w_equal = np.ones(N) / N                            # 等权对比基准

print(f"训练期: {log_rets_train.index[0].strftime('%Y-%m-%d')} ~ {log_rets_train.index[-1].strftime('%Y-%m-%d')}")
print(f"最小方差组合权重: 均值={w_minvar.mean():.4f}, 标准差={w_minvar.std():.4f}")
print(f"  正权重比例={np.sum(w_minvar>0)/N:.1%}, 负权重比例={np.sum(w_minvar<0)/N:.1%}")

# 样本外测试
df_test = df[df['time'] >= '2025-06-01'].copy()
pivot_test = df_test.pivot(index='time', columns='thscode', values='close')
log_rets_test = np.log(pivot_test / pivot_test.shift(1)).dropna()
R_test = log_rets_test.values

rp_minvar = R_test @ w_minvar                          # 样本外组合收益率
rp_equal  = R_test @ w_equal

def metrics(rp):
    a, v = np.mean(rp)*252, np.std(rp)*np.sqrt(252)
    cum = np.exp(np.cumsum(rp))
    dd = np.min(cum / np.maximum.accumulate(cum) - 1)
    return a, v, a/v, dd, cum

ret_mv, vol_mv, sr_mv, dd_mv, cum_mv = metrics(rp_minvar)
ret_eq, vol_eq, sr_eq, dd_eq, cum_eq = metrics(rp_equal)

print(f"\n样本外测试 ({log_rets_test.index[0].strftime('%Y-%m-%d')} ~ {log_rets_test.index[-1].strftime('%Y-%m-%d')}):")
print(f"{'策略':<12} {'年化收益':>8} {'年化波动':>8} {'夏普比率':>8} {'最大回撤':>8}")
print(f"{'最小方差':<12} {ret_mv:>8.2%} {vol_mv:>8.2%} {sr_mv:>8.2f} {dd_mv:>8.2%}")
print(f"{'等权':<12} {ret_eq:>8.2%} {vol_eq:>8.2%} {sr_eq:>8.2f} {dd_eq:>8.2%}")
print(f"{'差异':<12} {ret_mv-ret_eq:>+7.2%} {vol_mv-vol_eq:>+7.2%} {sr_mv-sr_eq:>+7.2f} {dd_mv-dd_eq:>+7.2%}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes[0].plot(log_rets_test.index, cum_mv, label='最小方差组合', color='#E74C3C', linewidth=1.5)
axes[0].plot(log_rets_test.index, cum_eq, label='等权组合', color='#2980B9', linewidth=1.5)
axes[0].set_title('样本外净值曲线: 最小方差 vs 等权', fontsize=13)
axes[0].set_xlabel('日期'); axes[0].set_ylabel('累计净值')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].bar(np.arange(N)-0.15, w_minvar, 0.3, label='最小方差权重', color='#E74C3C', alpha=0.8)
axes[1].bar(np.arange(N)+0.15, w_equal, 0.3, label='等权权重', color='#2980B9', alpha=0.8)
axes[1].axhline(y=0, color='black', linewidth=0.5)
axes[1].set_title('权重分布对比', fontsize=13)
axes[1].set_xlabel('股票序号'); axes[1].set_ylabel('权重')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/ch14_fig5_minvar_portfolio.png', dpi=150, bbox_inches='tight')
plt.show()

**运行结果**:
```
训练期: 2024-06-04 ~ 2025-05-30
最小方差组合权重: 均值=0.0200, 标准差=0.0843
  正权重比例=58.0%, 负权重比例=42.0%

样本外测试 (2025-06-04 ~ 2026-05-29):
策略            年化收益     年化波动     夏普比率     最大回撤
最小方差         12.99%    13.53%     0.96   -14.89%
等权             6.24%    17.40%     0.36   -19.09%
差异           +6.75%    -3.87%   +0.60    +4.21%
```

> **关键收获**: 最小方差组合在样本外将波动率从17.40%降至13.53%（降低3.87个百分点），夏普比率从0.36提升至0.96。但注意——最小方差组合有42%的权重为负（做空），在实际执行中需要考虑融券成本和做空约束。**数学给出的是"无摩擦最优"，工程落地需要加入现实约束。**

---

## 14.5 协方差矩阵: 从数据到估计——量化实践的完整流程

### 14.5.1 样本协方差矩阵的计算

给定 $T \times N$ 的去均值收益率矩阵 $\tilde{\mathbf{R}}$ (每列均值已归零):

$$\boxed{\mathbf{S} = \frac{1}{T-1} \tilde{\mathbf{R}}^T \tilde{\mathbf{R}}}$$

$\mathbf{S}$ 是 $N \times N$ 的对称矩阵。对角线元素 $S_{ii} = \sigma_i^2$ (第 $i$ 只资产的方差)，非对角线元素 $S_{ij} = \hat{\sigma}_{ij}$ (资产 $i$ 与 $j$ 的协方差)。

### 14.5.2 样本协方差矩阵的致命问题——以及量化行业的解决方案

当 $N$ (资产数) 与 $T$ (样本量) 可比时，样本协方差矩阵 $\mathbf{S}$ 的估计误差极大。极端情况: 当 $N > T$ 时，$\mathbf{S}$ 甚至不可逆。

量化行业的标准解决方案是**Ledoit-Wolf 收缩估计**:

$$\boxed{\mathbf{\Sigma}_{\text{LW}} = (1 - \delta) \mathbf{S} + \delta \mathbf{F}}$$

其中:
- $\mathbf{S}$: 样本协方差矩阵(无偏但方差大)
- $\mathbf{F}$: 结构化的"靶标"矩阵(有偏但方差小)——最简单的靶标是"常数相关矩阵"
- $\delta \in [0,1]$: 收缩强度，由数据驱动选择(最小化 Frobenius 范数下的期望误差)

**直觉**: 收缩估计 = 样本估计 + 先验信息的加权平均。样本不足时，向简单结构"收缩"可以大幅提高样本外表现。在典型的多因子模型中，$\delta$ 通常在 0.2-0.4 之间。

### 14.5.3 正半定性: 方差不能为负的数学保证

协方差矩阵一个至关重要的性质是**正半定性 (Positive Semi-Definite)**:

$$\boxed{\mathbf{w}^T \mathbf{\Sigma} \mathbf{w} \geq 0, \quad \forall \mathbf{w} \in \mathbb{R}^N}$$

**金融含义**: 任何非零权重向量的组合方差 $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$ 必须 $\geq 0$——因为方差不能是负数。如果矩估计出一个"负方差"的组合，说明估计过程出了严重问题(如当 $N > T$ 时的样本协方差矩阵)。

> **量化实践——过滤负特征值**: 在组合优化前，检查 $\mathbf{\Sigma}$ 的特征值。如果存在负特征值(哪怕只是 $-10^{-12}$ 量级的浮点误差)，将其截断为 $10^{-8}$ 或使用"高维协方差矩阵的最近正定校正"(Higham 算法)。

### 14.5.4 量化实战: Ledoit-Wolf 收缩——行业标准的协方差估计

用50只A股真实数据，对比样本协方差和LW收缩估计的差异:

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

df_recent = df[df['time'] >= '2024-06-01'].copy()
pivot_close = df_recent.pivot(index='time', columns='thscode', values='close')
log_rets = np.log(pivot_close / pivot_close.shift(1)).dropna()

R = log_rets.values; T, N = R.shape
R_demean = R - R.mean(axis=0)

# 样本协方差
S_sample = (R_demean.T @ R_demean) / (T - 1)

# Ledoit-Wolf 收缩 —— sklearn 自动选择最优 delta
from sklearn.covariance import LedoitWolf
lw = LedoitWolf()
lw.fit(R_demean)
S_lw = lw.covariance_
delta = lw.shrinkage_

print(f"Ledoit-Wolf 收缩强度 delta = {delta:.4f} "
      f"({1-delta:.1%} 样本 + {delta:.1%} 先验)")

# 对比特征值
ev_sample = np.sort(np.linalg.eigvalsh(S_sample))[::-1]
ev_lw = np.sort(np.linalg.eigvalsh(S_lw))[::-1]
print(f"条件数: 样本={ev_sample[0]/ev_sample[-1]:.0f}, LW={ev_lw[0]/ev_lw[-1]:.0f}")

# 可视化
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# 左图: 协方差矩阵热力图, 按行业排序
industry_order = [df[df['thscode']==c]['industry'].iloc[0] for c in log_rets.columns]
order_idx = np.argsort(industry_order)
S_sorted = S_sample[order_idx][:, order_idx]
im1 = axes[0].imshow(S_sorted, cmap='RdBu_r', aspect='auto', vmin=-0.0005, vmax=0.0005)
axes[0].set_title(f'样本协方差矩阵 ({N}只股票, 按行业排序)', fontsize=13)
axes[0].set_xlabel('股票 (按行业分组)'); axes[0].set_ylabel('股票 (按行业分组)')
plt.colorbar(im1, ax=axes[0], shrink=0.8)

# 右图: 特征值对比
x_ev = np.arange(1, 21)
axes[1].bar(x_ev-0.15, ev_sample[:20], 0.3, label='样本估计', color='#E74C3C', alpha=0.8)
axes[1].bar(x_ev+0.15, ev_lw[:20], 0.3, label=f'LW收缩 (delta={delta:.3f})',
            color='#2980B9', alpha=0.8)
axes[1].set_title('前20个特征值: 样本 vs LW收缩', fontsize=13)
axes[1].set_xlabel('特征值序号'); axes[1].set_ylabel('特征值')
axes[1].legend(); axes[1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/ch14_fig3_covariance_shrinkage.png', dpi=150, bbox_inches='tight')
plt.show()

**运行结果**:
```
Ledoit-Wolf 收缩强度 delta = 0.0515 (94.9% 样本 + 5.1% 先验)
条件数: 样本=519, LW=233
```

> **关键收获**: 即使只有5.1%的"收缩向先验"，条件数就从519降至233——矩阵稳定性提升了一倍以上。热力图清晰地展示了**行业聚类结构**: 对角线上的亮色方块对应同一行业内股票之间的高协方差。这就是为什么风险模型必须考虑行业因子——忽略行业结构会严重低估组合的真实风险。

---

## 14.6 特征值与特征向量: 寻找矩阵的"不变方向"

### 14.6.1 问题: 矩阵作用在向量上, 向量的方向会变吗?

第14.2节学过: 矩阵 $\mathbf{A}$ 左乘向量 $\mathbf{x}$，得到一个新向量 $\mathbf{A}\mathbf{x}$。一般来说，$\mathbf{A}\mathbf{x}$ 与 $\mathbf{x}$ 的**方向和长度都不相同**。

但存在一些**特殊方向**: 沿着这些方向的向量，被 $\mathbf{A}$ 左乘后，**方向不变**，只是长度被拉伸(或压缩)了一个倍数。用公式写就是:

$$\boxed{\mathbf{A}\mathbf{v} = \lambda \mathbf{v}}$$

这个方程叫做**特征方程 (Eigenvalue Equation)**。其中:
- $\mathbf{v}$: 满足"方向不变"的特殊向量 (且 $\mathbf{v} \neq \mathbf{0}$)
- $\lambda$: 描述"拉伸/压缩多少倍"的标量
- $\mathbf{A}$: 必须是方阵 ($N \times N$)，因为只有方阵才能把向量映射到同维空间

> **术语来源**: "Eigen" 是德语词，意为"自身的、特有的"。所以 Eigenvalue = 矩阵"自身固有的"拉伸倍数，Eigenvector = 矩阵"自身固有的"不变方向。中文译作"特征值"和"特征向量"——它们是这个矩阵的"指纹"，就像人的指纹一样独一无二。

### 14.6.2 一个微型例子: 手算2×2矩阵的特征值与特征向量

与其背诵定义，不如亲手算一遍。取一个最简单的 $2 \times 2$ 矩阵:

$$\mathbf{A} = \begin{pmatrix} 3 & 1 \\ 0 & 2 \end{pmatrix}$$

**Step 1——把特征方程改写为线性方程组**:

$$\mathbf{A}\mathbf{v} = \lambda \mathbf{v} \quad \Longrightarrow \quad \mathbf{A}\mathbf{v} - \lambda \mathbf{v} = \mathbf{0} \quad \Longrightarrow \quad (\mathbf{A} - \lambda \mathbf{I})\mathbf{v} = \mathbf{0}$$

其中 $\mathbf{I}$ 是单位矩阵，$\mathbf{A} - \lambda \mathbf{I} = \begin{pmatrix} 3-\lambda & 1 \\ 0 & 2-\lambda \end{pmatrix}$。

**Step 2——$\mathbf{v} \neq \mathbf{0}$ 的条件: $\det(\mathbf{A} - \lambda\mathbf{I}) = 0$**:

如果 $(\mathbf{A} - \lambda \mathbf{I})\mathbf{v} = \mathbf{0}$ 有非零解，则矩阵 $(\mathbf{A} - \lambda \mathbf{I})$ 必须是"不可逆"的——它的各列必须线性相关。用行列式表达这个条件:

$$\det(\mathbf{A} - \lambda \mathbf{I}) = (3-\lambda)(2-\lambda) - 1 \times 0 = 0$$

这个关于 $\lambda$ 的方程称为**特征方程 (Characteristic Equation)**:

$$(3-\lambda)(2-\lambda) = 0 \quad \Longrightarrow \quad \lambda_1 = 3, \quad \lambda_2 = 2$$

**这就是 $\mathbf{A}$ 的两个特征值**: 3 和 2。

**Step 3——对每个特征值，找到对应的特征向量**:

对于 $\lambda_1 = 3$: 代入 $(\mathbf{A} - 3\mathbf{I})\mathbf{v} = \mathbf{0}$:

$$\begin{pmatrix} 0 & 1 \\ 0 & -1 \end{pmatrix} \begin{pmatrix} v_1 \\ v_2 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \end{pmatrix} \quad \Longrightarrow \quad v_2 = 0, \; v_1 \text{ 任意}$$

取 $\mathbf{v}_1 = \begin{pmatrix} 1 \\ 0 \end{pmatrix}$ (或它的任意非零倍数)。

对于 $\lambda_2 = 2$: 代入 $(\mathbf{A} - 2\mathbf{I})\mathbf{v} = \mathbf{0}$:

$$\begin{pmatrix} 1 & 1 \\ 0 & 0 \end{pmatrix} \begin{pmatrix} v_1 \\ v_2 \end{pmatrix} = \begin{pmatrix} 0 \\ 0 \end{pmatrix} \quad \Longrightarrow \quad v_1 + v_2 = 0$$

取 $\mathbf{v}_2 = \begin{pmatrix} 1 \\ -1 \end{pmatrix}$。

**Step 4——验证**: 

$$\mathbf{A}\mathbf{v}_1 = \begin{pmatrix} 3 & 1 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} 1 \\ 0 \end{pmatrix} = \begin{pmatrix} 3 \\ 0 \end{pmatrix} = 3 \cdot \begin{pmatrix} 1 \\ 0 \end{pmatrix} = 3\mathbf{v}_1 \; \checkmark$$

$$\mathbf{A}\mathbf{v}_2 = \begin{pmatrix} 3 & 1 \\ 0 & 2 \end{pmatrix} \begin{pmatrix} 1 \\ -1 \end{pmatrix} = \begin{pmatrix} 2 \\ -2 \end{pmatrix} = 2 \cdot \begin{pmatrix} 1 \\ -1 \end{pmatrix} = 2\mathbf{v}_2 \; \checkmark$$

> **关键观察**: $\mathbf{v}_2 = (1, -1)^T$ 被 $\mathbf{A}$ 左乘后，方向完全不变——仍然指向"右下方"45°——只是长度变成了原来的2倍。而随便取一个不是特征向量的向量，比如 $\mathbf{x} = (1, 1)^T$，算 $\mathbf{A}\mathbf{x} = (4, 2)^T$——跟原来的 $(1, 1)^T$ 相比，方向明显改变了。**这就是特征向量的独特之处: 被矩阵作用后，方向不走样。**

### 14.6.3 特征值的三个基本事实

从上面的推导，可以总结出三个关键事实:

**事实1——$N \times N$ 矩阵有 $N$ 个特征值 (计重数)**。特征方程 $\det(\mathbf{A} - \lambda\mathbf{I}) = 0$ 是一个 $N$ 次多项式方程，根据代数基本定理，它有 $N$ 个根(可能有重复、可能是复数)。对于协方差矩阵这种对称矩阵，所有特征值保证是**实数**且**非负**——这正是它适合做风险分解的数学原因。

**事实2——特征向量只确定"方向"，长度任意**。如果 $\mathbf{v}$ 是特征向量，则 $c\mathbf{v}$ ($c \neq 0$) 也是。所以我们通常把特征向量**归一化为单位长度** ($\|\mathbf{v}\| = 1$)，只保留方向信息。

**事实3——不同特征值对应的特征向量线性无关**。在上例中，$\mathbf{v}_1 = (1,0)^T$ 和 $\mathbf{v}_2 = (1,-1)^T$ 明显不共线。对于对称矩阵，更进一步: 不同特征值对应的特征向量互相**正交** ($\mathbf{v}_i^T \mathbf{v}_j = 0$)。这个性质在金融中极其重要——它意味着不同的"风险方向"彼此独立。

### 14.6.4 几何直觉: 矩阵 = 空间的"拉伸机"

![特征向量的几何直觉: 矩阵 A 作用在一般向量上会旋转方向, 但作用在特征向量 v 上只拉伸不旋转, 拉伸倍数为特征值 lambda。](images/ch14_fig1_eigenvector_intuition.png)

想象把平面上的所有点看作从原点出发的向量。矩阵 $\mathbf{A}$ 左乘就是把每个向量"映射"到新位置。大多数向量在映射过程中既被旋转又被拉伸——但特征向量所在的直线是**不转的**: 整条直线上所有点只是被均匀拉伸(或压缩)了 $\lambda$ 倍。

如果把矩阵想象成一台"拉伸机": 它沿着特征向量的方向拉伸空间，拉伸倍数就是特征值。特征值越大，那个方向被拉得越长；特征值 $\approx 0$，那个方向几乎被压扁。

这个几何图像直接通向金融: **协方差矩阵 $\mathbf{\Sigma}$ 在"风险空间"中沿着某些方向拉伸得特别厉害(高特征值 = 高系统风险方向)，沿着另一些方向几乎不拉伸(低特征值 = 可分散的方向)**。

### 14.6.5 协方差矩阵的特征值 = 风险方向的方差

把上面的所有知识应用到 $\mathbf{A} = \mathbf{\Sigma}$ (协方差矩阵):

将 $\mathbf{\Sigma}\mathbf{v} = \lambda\mathbf{v}$ 两边左乘 $\mathbf{v}^T$:

$$\mathbf{v}^T \mathbf{\Sigma} \mathbf{v} = \lambda \mathbf{v}^T \mathbf{v} = \lambda \|\mathbf{v}\|^2$$

如果 $\mathbf{v}$ 是单位向量 ($\|\mathbf{v}\|=1$)，则 $\lambda = \mathbf{v}^T \mathbf{\Sigma} \mathbf{v}$。

**金融解读——这就是"组合方差"!** 第9章学过，$\mathbf{w}^T\mathbf{\Sigma}\mathbf{w}$ 是权重为 $\mathbf{w}$ 的组合方差。如果把单位特征向量 $\mathbf{v}$ 当作权重，$\mathbf{v}^T \mathbf{\Sigma} \mathbf{v}$ 恰好是这个"特征向量组合"的方差。所以:

$$\boxed{\text{特征值 } \lambda_i = \text{以第 } i \text{ 特征向量为权重的组合方差}}$$

$$\boxed{\text{特征向量 } \mathbf{v}_i = \text{协方差矩阵的一个"独立风险方向"}}$$

各特征向量之间是**正交**的 ($\mathbf{v}_i^T \mathbf{v}_j = 0, i \neq j$)，这意味着: **不同风险方向上的收益率"互不投影"，风险可以按方向独立加总**。总风险 $\text{tr}(\mathbf{\Sigma}) = \sum \lambda_i$ 被完美分解为 $N$ 个互不重叠的方向。

### 14.6.6 特征值从大到小排序 = 风险从系统到特异

将特征值从大到小排列: $\lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_N \geq 0$。

- $\lambda_1$ (最大值): 最大的"风险方向"——通常对应**市场因子**。$\mathbf{v}_1$ 的各个分量往往同号(全部正或全部负)，即一个"几乎等权"的组合。在典型的A股数据中(见14.7.5的代码运行结果)，50只股票的第一特征向量49个分量为正、仅1个为负。
- $\lambda_1 / \sum \lambda_i$: **第一主成分解释的风险占比**。在A股市场中，这个比例通常在 30-50%——意味着"整个市场一起涨跌"贡献了总风险的三分之一到一半。
- $\lambda_N$ (最小值): 最小的独立风险方向——对应可以被分散化几乎消除的方向。

> **量化实践——PCA风险模型的"一句话总结"**: 前 $K$ 个特征值之和 $\div$ 总特征值之和 = 前 $K$ 个主成分解释的风险比例。如果前3个特征值解释了超过50%的总方差(如14.7.5的例子中前三主成分=58.9%)，说明这是一个"高系统性风险"的市场；如果前10个才解释50%，说明个股特异性风险占主导，分散化空间更大。

---

## 14.7 特征分解与主成分: 量化风险管理的标准工具

### 14.7.1 特征分解定理

对称矩阵 $\mathbf{\Sigma}$ 的特征分解 (Eigendecomposition)，也称谱分解:

$$\boxed{\mathbf{\Sigma} = \mathbf{Q} \mathbf{\Lambda} \mathbf{Q}^T = \sum_{i=1}^{N} \lambda_i \mathbf{v}_i \mathbf{v}_i^T}$$

其中:
- $\mathbf{Q} = [\mathbf{v}_1, \mathbf{v}_2, \dots, \mathbf{v}_N]$: $N \times N$ 正交矩阵，列是单位特征向量
- $\mathbf{\Lambda} = \text{diag}(\lambda_1, \dots, \lambda_N)$: 对角矩阵，对角线是特征值
- $\mathbf{Q}^T\mathbf{Q} = \mathbf{I}$: 正交性保证 ($\mathbf{Q}^{-1} = \mathbf{Q}^T$)

### 14.7.2 特征组合 (Eigenportfolio): 从抽象数学到可交易组合

特征向量 $\mathbf{v}_i$ 的分量可正可负，且不一定满足 $\sum v_{ij} = 1$ (权重之和为1的约束)。但它定义了 $N$ 个**正交的风险方向**:

- **$\mathbf{v}_1$ (第一特征组合)**: 各分量通常接近等权，代表"市场因子"——这是纯粹的 Beta 暴露
- **$\mathbf{v}_2$ (第二特征组合)**: 呈现"行业分化"或"规模分化"——一半正一半负的多空结构
- **高序号特征组合**: 对应噪音方向——信噪比极低，不应作为交易信号

研究实践 (Avellaneda & Lee, 2010) 表明: 对去均值收益率做主成分分解后，前几个特征组合的多空收益具有统计显著性——这构成了**统计套利**中"中性化"策略的基础。

### 14.7.3 特征值衰减速度 = 市场有效性度量

定义累积风险解释比例:

$$R^2_K = \frac{\sum_{i=1}^{K} \lambda_i}{\sum_{i=1}^{N} \lambda_i}$$

$R^2_K$ 随 $K$ 增长的速度反映了市场的"因子结构强度":

| 市场特征 | $R^2_3$ 典型值 | $R^2_1$ 典型值 | 分散化难度 |
|---------|--------------|--------------|-----------|
| 高系统性(如A股) | 50-65% | 35-50% | 较难——大部分风险来自共同因子 |
| 中系统性(如美股) | 25-40% | 15-25% | 适中——个股特异性空间较大 |
| 低系统性(如加密货币) | 15-25% | 10-15% | 较易——但特异性波动也更高 |

> **量化应用——确定风险模型的因子数量**: 在实际 Barra/Axioma 等商业风险模型中，统计风险因子(区别于预先定义的行业/风格因子)的数量通常通过"特征值大于某个阈值"或"累积解释比例 > 某值"来确定。典型的A股统计风险模型使用 10-20 个主成分。

### 14.7.4 从特征分解到组合优化: $\mathbf{\Sigma}^{-1}$ 的主成分表达

特征分解给出了 $\mathbf{\Sigma}^{-1}$ 的精确表达式:

$$\boxed{\mathbf{\Sigma}^{-1} = \mathbf{Q} \mathbf{\Lambda}^{-1} \mathbf{Q}^T = \sum_{i=1}^{N} \frac{1}{\lambda_i} \mathbf{v}_i \mathbf{v}_i^T}$$

这个公式揭示了一个深刻的事实: **最小方差组合会严重低配高特征值方向(高系统风险方向)、高配低特征值方向(低风险方向)**。因为 $\mathbf{\Sigma}^{-1}$ 中，每个方向被其方差的倒数 $1/\lambda_i$ 加权——方差越大的方向，在逆矩阵中的权重越小。

但这同时也是最小方差组合的脆弱性来源: 最小的特征值 $\lambda_N$ 估计误差最大，而 $\mathbf{\Sigma}^{-1}$ 中 $1/\lambda_N$ 放大了这个误差。这就是为什么实践中通常需要**收缩**或用**因子模型**来正则化协方差矩阵的估计。

### 14.7.5 量化实战: PCA风险分解——用特征值透视市场结构

下面的代码对50只A股的协方差矩阵做完整的特征分解，回答三个量化核心问题: (1) 多少个主成分解释了80%的风险？ (2) 第一特征向量是否对应"市场因子"？ (3) 特征值衰减速度揭示了什么市场结构？

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['WenQuanYi Micro Hei']
matplotlib.rcParams['axes.unicode_minus'] = False

csv_path = 'data/stock_data_50_20210601_20260531.csv'
df = pd.read_csv(csv_path, parse_dates=['time'])

df_recent = df[df['time'] >= '2024-06-01'].copy()
pivot_close = df_recent.pivot(index='time', columns='thscode', values='close')
log_rets = np.log(pivot_close / pivot_close.shift(1)).dropna()

R = log_rets.values; T, N = R.shape
R_demean = R - R.mean(axis=0)
Sigma = (R_demean.T @ R_demean) / (T - 1)

# 特征分解 (eigh 专用于对称矩阵, 保证实数输出)
eigvals, eigvecs = np.linalg.eigh(Sigma)
eigvals = eigvals[::-1]; eigvecs = eigvecs[:, ::-1]  # 降序排列

var_explained = eigvals / np.sum(eigvals)
cum_var = np.cumsum(var_explained)
n_80 = np.searchsorted(cum_var, 0.80) + 1

print(f"前10个特征值: {eigvals[:10].round(4)}")
print(f"\n解释方差比例 (前10个主成分):")
for i in range(10):
    print(f"  PC{i+1}: {var_explained[i]:.2%} (累计 {cum_var[i]:.2%})")
print(f"\n需要 {n_80} 个主成分解释 80% 总风险")

v1, v2 = eigvecs[:, 0], eigvecs[:, 1]
print(f"\n第一特征向量: 正分量={np.sum(v1>0)}/{N}, "
      f"均值={v1.mean():.3f} -> '市场因子' (几乎全员同向)")
print(f"第二特征向量: 正分量={np.sum(v2>0)}/{N}, "
      f"范围=[{v2.min():.3f}, {v2.max():.3f}] -> '分化因子' (多空结构)")

# 可视化: 2x2 面板
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes[0,0].plot(range(1,N+1), eigvals, 'o-', color='#2980B9', markersize=4, linewidth=1.5)
axes[0,0].set_title('特征值衰减 (碎石图)', fontsize=13)
axes[0,0].set_xlabel('主成分序号'); axes[0,0].set_ylabel('特征值')
axes[0,0].axvline(x=10, color='red', linestyle='--', alpha=0.5)
axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(range(1,N+1), cum_var*100, 'o-', color='#27AE60', markersize=4, linewidth=1.5)
axes[0,1].axhline(y=80, color='red', linestyle='--', alpha=0.5)
axes[0,1].axvline(x=n_80, color='red', linestyle='--', alpha=0.5)
axes[0,1].set_title(f'累积风险解释 (80%需{n_80}个PC)', fontsize=13)
axes[0,1].set_xlabel('主成分数量'); axes[0,1].set_ylabel('累积解释比例 (%)')
axes[0,1].set_ylim(0,105); axes[0,1].grid(True, alpha=0.3)

colors_v1 = ['#E74C3C' if v>0 else '#2980B9' for v in v1]
axes[1,0].bar(range(N), v1, color=colors_v1, alpha=0.8, edgecolor='none')
axes[1,0].axhline(y=0, color='black', linewidth=0.5)
axes[1,0].set_title(f'第一特征向量 ({var_explained[0]:.1%}风险) —— 市场因子', fontsize=13)
axes[1,0].set_xlabel('股票序号'); axes[1,0].set_ylabel('分量')
axes[1,0].grid(True, alpha=0.3, axis='y')

colors_v2 = ['#E74C3C' if v>0 else '#2980B9' for v in v2]
axes[1,1].bar(range(N), v2, color=colors_v2, alpha=0.8, edgecolor='none')
axes[1,1].axhline(y=0, color='black', linewidth=0.5)
axes[1,1].set_title(f'第二特征向量 ({var_explained[1]:.1%}风险) —— 分化因子', fontsize=13)
axes[1,1].set_xlabel('股票序号'); axes[1,1].set_ylabel('分量')
axes[1,1].grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('images/ch14_fig4_eigendecomposition.png', dpi=150, bbox_inches='tight')
plt.show()

**运行结果**:
```
前10个特征值: [0.0145 0.0022 0.002 0.0011 0.0009 0.0008 0.0007 0.0007 0.0006 0.0005]

解释方差比例 (前10个主成分):
  PC1: 45.64% (累计 45.64%)
  PC2: 6.97% (累计 52.61%)
  PC3: 6.28% (累计 58.89%)
  ...
  PC10: 1.71% (累计 75.75%)

需要 13 个主成分解释 80% 总风险

第一特征向量: 正分量=49/50, 均值=0.127 -> '市场因子' (几乎全员同向)
第二特征向量: 正分量=15/50, 范围=[-0.193, 0.449] -> '分化因子' (多空结构)
```

> **量化解读**: PC1 独自解释了45.64%的风险——这意味着A股市场中，**将近一半的个股波动可以用"整个市场一起涨跌"这一个因素来解释**。这正是为什么简单的等权指数对冲(买入个股、做空指数期货)能在很大程度上降低组合波动。前3个主成分合计解释58.89%，13个主成分才到80%——说明剩余的"特异性风险"分布较为分散，需要相当数量的统计因子才能有效捕捉。

---

## 14.8 核心公式速查

> 本节是前述各节公式的集中汇总, 供复习和查阅使用.

| 概念 | 公式 | 量化意义 |
|------|------|---------|
| 矩阵乘法(元素) | $C_{ij} = \sum_k A_{ik} B_{kj}$ | 行×列的点积——矩阵计算的原子操作 |
| 矩阵乘法(列视角) | $\mathbf{R}\mathbf{W} = [\mathbf{R}\mathbf{w}_1 \ \cdots \ \mathbf{R}\mathbf{w}_p]$ | 批量回测: K个策略×一个矩阵乘法 |
| 因子归因 | $\mathbf{R}_{\text{factor}} = \mathbf{F} \mathbf{B}^T$ | 分解收益为"因子解释部分"+"特异部分" |
| 乘积转置 | $(\mathbf{A}\mathbf{B})^T = \mathbf{B}^T \mathbf{A}^T$ | 协方差矩阵推导的基础恒等式 |
| 协方差矩阵 | $\mathbf{S} = \frac{1}{T-1}\tilde{\mathbf{R}}^T\tilde{\mathbf{R}}$ | 从收益率矩阵一步算出协方差矩阵 |
| Ledoit-Wolf收缩 | $\mathbf{\Sigma}_{\text{LW}} = (1-\delta)\mathbf{S} + \delta\mathbf{F}$ | 样本外风险预测更准——量化行业标配 |
| 逆矩阵 | $\mathbf{A}\mathbf{A}^{-1} = \mathbf{I}$ | "撤销"线性变换——OLS和组合优化的核心 |
| 最小方差组合 | $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1} / (\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$ | 给定协方差矩阵的理论最优低风险组合 |
| 特征方程 | $\mathbf{\Sigma}\mathbf{v} = \lambda\mathbf{v}$ | 寻找"不变方向"——风险结构的主轴 |
| 特征值=方差 | $\lambda_i = \mathbf{v}_i^T\mathbf{\Sigma}\mathbf{v}_i$ | 第i个风险方向的"方差大小" |
| 特征分解 | $\mathbf{\Sigma} = \mathbf{Q}\mathbf{\Lambda}\mathbf{Q}^T$ | 协方差矩阵的完全分解——PCA的数学基础 |
| 累积解释比例 | $R^2_K = \sum_{i=1}^K \lambda_i / \sum \lambda_i$ | 前K个主成分解释了总风险的百分之几 |
| 逆矩阵的特征分解 | $\mathbf{\Sigma}^{-1} = \sum \lambda_i^{-1} \mathbf{v}_i\mathbf{v}_i^T$ | 揭示最小方差组合的本质: 低配高风险方向 |
| 条件数 | $\kappa = \lambda_{\max}/\lambda_{\min}$ | >30时协方差矩阵接近不可逆——需收缩或降维 |
| 精度矩阵 | $\mathbf{P} = \mathbf{\Sigma}^{-1}$ | 偏相关系数——"控制其他所有变量后的净相关" |

---

## 14.9 本章小结

| 概念 | 核心公式 | 量化意义 |
|------|---------|---------|
| 矩阵乘法(列视角) | $\mathbf{R}\mathbf{W} = [\mathbf{R}\mathbf{w}_1 \cdots \mathbf{R}\mathbf{w}_p]$ | 批量回测: K个策略一次完成 |
| 因子归因 | $\mathbf{R}_{\text{factor}} = \mathbf{F}\mathbf{B}^T$ | 收益分解: 因子部分 + 特异部分 |
| 协方差矩阵 | $\mathbf{S} = \frac{1}{T-1}\tilde{\mathbf{R}}^T\tilde{\mathbf{R}}$ | 从收益率数据到风险矩阵的一步计算 |
| Ledoit-Wolf收缩 | $\mathbf{\Sigma}_{\text{LW}} = (1-\delta)\mathbf{S} + \delta\mathbf{F}$ | 行业标配: 样本外风险预测更准 |
| 逆矩阵 | $\mathbf{A}^{-1}\mathbf{A} = \mathbf{I}$ | OLS: $(\mathbf{X}^T\mathbf{X})^{-1}$; 组合优化: $\mathbf{\Sigma}^{-1}\mathbf{1}$ |
| 精度矩阵 | $\mathbf{P} = \mathbf{\Sigma}^{-1}$ | 偏相关系数——控制其他变量后的"净相关" |
| 特征方程 | $\mathbf{\Sigma}\mathbf{v} = \lambda\mathbf{v}$ | 寻找风险"不变方向" |
| 特征值=组合方差 | $\lambda_i = \mathbf{v}_i^T\mathbf{\Sigma}\mathbf{v}_i$ | 第i个风险方向的波动率平方 |
| 特征分解 | $\mathbf{\Sigma} = \mathbf{Q}\mathbf{\Lambda}\mathbf{Q}^T$ | 协方差矩阵的完全分解 |
| 累积解释比 | $R^2_K = \sum_1^K \lambda_i / \sum \lambda_i$ | K个主成分解释了多少系统性风险 |
| 条件数 | $\kappa = \lambda_{\max}/\lambda_{\min}$ | >30 时需收缩或降维 |
| 最小方差组合 | $\mathbf{w}_{\min} = \mathbf{\Sigma}^{-1}\mathbf{1} / (\mathbf{1}^T\mathbf{\Sigma}^{-1}\mathbf{1})$ | 给定协方差的理论最优低风险组合 |

**从本章走向下一章**:
- 第15章将用本章的特征分解和逆矩阵概念，从"线性变换"和"正交投影"的几何视角重新推导OLS——你会看到 $\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$ 不仅是"最小化残差平方和的解"，更是"$\mathbf{y}$ 在 $\mathbf{X}$ 列空间上的正交投影"
- 第16-17章将把本章的逆矩阵和特征分解应用到投资组合优化——均值-方差前沿、风险预算、Black-Litterman 模型的完整推导

---

## 14.10 练习题

### 数学推导

**题1——矩阵乘法的维度约束与金融含义**: 

(a) 设 $\mathbf{R}$ 是 $T \times N$ 收益率矩阵，$\mathbf{W}_1$ 是 $N \times K$ 的权重矩阵。验证 $\mathbf{R}\mathbf{W}_1$ 可乘且结果是 $T \times K$。如果 $\mathbf{W}_2$ 是 $K \times M$ 矩阵，$(\mathbf{R}\mathbf{W}_1)\mathbf{W}_2$ 的维度是多少？解释这个运算在"先算K个策略、再对策略做M种组合"的量化场景中的含义。

(b) 为什么 $\mathbf{W}\mathbf{R}$ (权重矩阵在前，收益率矩阵在后) 在维度上通常不可行？什么条件下才可行？

**题2——协方差矩阵的对称性与正半定性**:

(a) 设 $\mathbf{\Sigma} = \frac{1}{T-1} \tilde{\mathbf{R}}^T \tilde{\mathbf{R}}$。证明 $\mathbf{\Sigma}^T = \mathbf{\Sigma}$。

(b) 对于任意 $\mathbf{w} \in \mathbb{R}^N$，证明 $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w} = \frac{1}{T-1} \|\tilde{\mathbf{R}} \mathbf{w}\|^2 \geq 0$。这个不等式的金融含义是什么？

(c) 在什么条件下 $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w} = 0$？这意味着什么？(提示: 考虑资产之间存在完全线性关系的情况。)

**题3——特征值分解与风险预算**:

(a) 对于协方差矩阵的特征分解 $\mathbf{\Sigma} = \sum_{i=1}^N \lambda_i \mathbf{v}_i \mathbf{v}_i^T$，证明组合方差 $\mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$ 可以分解为 $\sum_{i=1}^N \lambda_i (\mathbf{w}^T \mathbf{v}_i)^2$。解释 $(\mathbf{w}^T \mathbf{v}_i)^2$ 的金融含义。

(b) 最小方差组合的性质: 证明 $\mathbf{w}_{\min}^T \mathbf{v}_i \propto 1/\lambda_i$ (即: 最小方差组合在各特征向量方向上的暴露与对应特征值成反比)。为什么这使得最小方差组合在实际中对 $\lambda_N$ 的估计误差极其敏感？

### 编程实践

**题4——批量回测与绩效归因**: 基于14.2.5的批量回测框架，扩展为以下分析:

(a) 除原有5个策略外，再添加两个策略: (i) 等风险贡献 (ERC) 近似——权重 $\propto 1/\sigma_i$ 且约束每个资产的风险贡献相等; (ii) 随机权重 (100组随机归一化权重取平均)。计算全部7个策略的夏普比率，并绘制夏普比率对比图。

(b) 对每个策略的日收益率序列，计算它与等权策略的**跟踪误差** (即差异收益率的标准差年化)。哪个策略的跟踪误差最大？哪个最小？解释为什么。

**题5——PCA 风险模型的构建与评估**: 

(a) 基于14.7.5的特征分解结果，构建一个"截断PCA协方差矩阵": $\mathbf{\Sigma}_{\text{PCA},K} = \sum_{i=1}^K \lambda_i \mathbf{v}_i \mathbf{v}_i^T$ (只保留前 $K$ 个特征值/特征向量，其余丢弃)。分别取 $K=3, 5, 10, 20$，计算:
- $\mathbf{\Sigma}_{\text{PCA},K}$ 与原始 $\mathbf{\Sigma}$ 的 Frobenius 范数相对误差
- 用 $\mathbf{\Sigma}_{\text{PCA},K}$ 计算的最小方差组合权重，与用原始 $\mathbf{\Sigma}$ 计算的权重的 $\ell_2$ 距离

(b) 哪个 $K$ 在"逼近精度"和"模型复杂度"之间取得了最好的平衡？画出误差随 $K$ 变化的曲线，并标注"肘部"位置。

---

## 14.11 参考文献

1. **Lay, D. C., Lay, S. R., & McDonald, J. J.** (2021). *Linear Algebra and Its Applications* (6th ed.). Pearson. （第5-7章系统讲解矩阵乘法、逆矩阵、特征值分解——本章数学框架的主要来源）

2. **Ledoit, O., & Wolf, M.** (2004). A well-conditioned estimator for large-dimensional covariance matrices. *Journal of Multivariate Analysis*, 88(2), 365-411. （Ledoit-Wolf收缩估计的原始论文——量化行业协方差矩阵估计的里程碑）

3. **Meucci, A.** (2005). *Risk and Asset Allocation*. Springer. （第4章用特征分解构建PCA风险模型; 第6章将逆矩阵与组合优化统一在贝叶斯框架下）

4. **Avellaneda, M., & Lee, J.-H.** (2010). Statistical arbitrage in the US equities market. *Quantitative Finance*, 10(7), 761-782. （PCA统计套利的经典实证研究——将特征向量组合应用于美股，展示前几个特征组合的超额收益）

5. **Grinold, R. C., & Kahn, R. N.** (1999). *Active Portfolio Management* (2nd ed.). McGraw-Hill. （第3章将协方差矩阵的估计、收缩和特征分解置于量化组合管理的全流程框架中）

6. **Glasserman, P.** (2004). *Monte Carlo Methods in Financial Engineering*. Springer. （第5章在金融数值计算的上下文中讨论矩阵运算和条件数——理解本章中"用 solve 而非 inv"的深层数值原因）

---

> **愿我们都能在数字与代码之间, 找到理解市场的那把钥匙.**
>
> *数学的理解没有捷径, 量化的能力无法外包.*